In [4]:
"""
AutoVault — Car Price Prediction Model
========================================
Reads from:
  data/merged_datasets/merged_luxe_dataset.csv
  data/merged_datasets/merged_ordinary_dataset.csv

Features used (auto-selected by importance):
  - tier          (luxury vs standard — strongest signal)
  - log_kms       (log of km driven — better than raw kms)
  - year          (model year)
  - brand         (target-encoded — mean price per brand)
  - fuel_type     (kept only if passes importance threshold)
  - transmission  (kept only if passes importance threshold)

Features REMOVED automatically:
  - kms_covered   (redundant — log_kms captures this better)
  - car_age       (redundant — same info as year, just flipped)
  - kms_per_year  (low importance, causes noise)
  - brand_enc     (label encoding brands is meaningless — replaced by target encoding)

Models:
  1. Random Forest       — baseline
  2. XGBoost (default)   — better accuracy
  3. XGBoost Fine-tuned  — Optuna hyperparameter search (BEST)

Output → models/
  xgb_finetuned.pkl, rf_baseline.pkl,
  encoders.pkl, feature_columns.pkl

SETUP:
  pip install pandas scikit-learn xgboost optuna matplotlib seaborn joblib

RUN:
  python train_price_model.py
"""

import os
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection   import train_test_split, KFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble          import RandomForestRegressor
from xgboost                   import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
LUXE_CSV  = "../../data/merged_datasets/merged_luxe_dataset.csv"
ORD_CSV   = "../../data/merged_datasets/merged_ordinary_dataset.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

CURRENT_YEAR         = 2026
OPTUNA_TRIALS        = 50       # increase to 100+ for better tuning
TEST_SIZE            = 0.20
RANDOM_STATE         = 42
IMPORTANCE_THRESHOLD = 0.01     # features below this are auto-dropped


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

def load_data() -> pd.DataFrame:
    print("=" * 60)
    print("  AutoVault — Car Price Prediction")
    print("=" * 60)
    print("\n📂 Loading CSVs…")

    lux = pd.read_csv(LUXE_CSV)
    lux = lux.drop(columns=[c for c in lux.columns if 'Unnamed' in c], errors='ignore')
    lux = lux.rename(columns={
        'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })
    lux['tier'] = 'luxury'

    ord_ = pd.read_csv(ORD_CSV)
    ord_ = ord_.drop(columns=[c for c in ord_.columns if 'Unnamed' in c], errors='ignore')
    ord_ = ord_.rename(columns={
        'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })
    ord_['tier'] = 'standard'

    needed = ['brand', 'price', 'kms_covered', 'year', 'fuel_type', 'transmission', 'tier']
    df = pd.concat([lux[needed], ord_[needed]], ignore_index=True)

    print(f"  Luxury   : {len(lux):>6,} rows")
    print(f"  Ordinary : {len(ord_):>6,} rows")
    print(f"  Combined : {len(df):>6,} rows")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# 2. CLEAN & ENGINEER FEATURES
# ═══════════════════════════════════════════════════════════════════════════

def clean_and_engineer(df: pd.DataFrame):
    print("\n🔧 Cleaning & engineering features…")
    df = df.copy()

    # Drop rows with missing target / key numerics
    before = len(df)
    df = df.dropna(subset=['price', 'kms_covered', 'year'])
    df = df[df['price'] > 0]
    df = df[df['kms_covered'] >= 0]
    print(f"  Dropped {before - len(df):,} rows with missing/invalid values")

    # Normalise text categories
    brand_map = {
        'Maruti Suzuki': 'Maruti',
        'Mercedes':      'Mercedes-Benz',
        'Rolls royce':   'Rolls-Royce',
        'KIA':           'Kia',
    }
    fuel_map = {'CNG & Hybrids': 'CNG'}

    df['brand']        = df['brand'].fillna('Unknown').str.strip().replace(brand_map)
    df['fuel_type']    = df['fuel_type'].fillna('Unknown').str.strip().replace(fuel_map)
    df['transmission'] = df['transmission'].fillna('Unknown').str.strip()
    df['tier']         = df['tier'].fillna('standard').str.strip()

    # Cast numerics
    df['year']        = pd.to_numeric(df['year'],        errors='coerce').fillna(2020).astype(int)
    df['kms_covered'] = pd.to_numeric(df['kms_covered'], errors='coerce').fillna(0).astype(float)
    df['price']       = pd.to_numeric(df['price'],       errors='coerce')
    df = df.dropna(subset=['price'])

    # Remove outliers (1st–99th percentile per tier)
    before = len(df)
    parts = []
    for tier in df['tier'].unique():
        sub = df[df['tier'] == tier].copy()
        lo, hi = sub['price'].quantile([0.01, 0.99])
        parts.append(sub[(sub['price'] >= lo) & (sub['price'] <= hi)])
    df = pd.concat(parts, ignore_index=True)
    print(f"  Removed {before - len(df):,} outliers → {len(df):,} rows remain")
    print(f"  Price range: ₹{df['price'].min():,.0f} – ₹{df['price'].max():,.0f}")

    # ── Target-encode brand (mean price per brand) ─────────────────────
    # This is vastly better than label encoding because it captures
    # that Ferrari listings are worth more than Maruti listings.
    brand_mean = df.groupby('brand')['price'].mean()
    df['brand_price_mean'] = df['brand'].map(brand_mean)
    brand_encoding = brand_mean.to_dict()

    # ── Label encode remaining categoricals ───────────────────────────
    encoders = {'brand_target': brand_encoding}
    for col in ['fuel_type', 'transmission', 'tier']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  {col}: {len(le.classes_)} classes → {list(le.classes_)}")

    # ── log_kms: log-transform compresses the huge kms range ──────────
    # log(1 + kms) is more informative than raw kms for price prediction
    df['log_kms'] = np.log1p(df['kms_covered'])

    # NOTE — deliberately excluded features (with reasoning):
    #   car_age      → perfectly redundant with year (r = -1.0 with year)
    #   kms_covered  → log_kms captures same info better (corr -0.40 vs -0.26)
    #   kms_per_year → correlation with price only -0.07, adds noise
    #   brand_enc    → label-encoding brands as integers is meaningless for trees

    print(f"  ✓ Feature engineering complete")
    return df, encoders, brand_encoding


# ═══════════════════════════════════════════════════════════════════════════
# 3. AUTOMATIC FEATURE SELECTION
#    Train a quick XGBoost on ALL candidate features, measure each
#    feature's importance score, auto-drop anything below threshold.
# ═══════════════════════════════════════════════════════════════════════════

CANDIDATE_FEATURES = [
    'brand_price_mean',   # target-encoded brand (best brand representation)
    'year',               # model year of the car
    'log_kms',            # log-transformed km driven
    'fuel_type_enc',      # petrol / diesel / electric / cng / hybrid
    'transmission_enc',   # manual / automatic
    'tier_enc',           # luxury vs standard (strongest single feature)
]


def select_features(X_train: pd.DataFrame, y_train: pd.Series):
    print("\n🔍 Automatic feature selection…")
    print(f"  Threshold: importance >= {IMPORTANCE_THRESHOLD}")

    # Quick XGBoost to measure importance
    probe = XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    probe.fit(X_train, y_train)

    importance = pd.Series(
        probe.feature_importances_,
        index=CANDIDATE_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  {'Feature':<22} {'Importance':>12}  Decision")
    print(f"  {'─' * 48}")
    kept, dropped = [], []
    for feat, imp in importance.items():
        decision = '✓  KEEP' if imp >= IMPORTANCE_THRESHOLD else '✗  DROP (below threshold)'
        print(f"  {feat:<22} {imp:>12.4f}  {decision}")
        if imp >= IMPORTANCE_THRESHOLD:
            kept.append(feat)
        else:
            dropped.append(feat)

    print(f"\n  → Kept    ({len(kept)}): {kept}")
    print(f"  → Dropped ({len(dropped)}): {dropped}")
    return kept, dropped, importance


# ═══════════════════════════════════════════════════════════════════════════
# 4. EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(model, X_test, y_test, name="Model"):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = math.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / y_test.clip(1))) * 100

    bar = '─' * 46
    print(f"\n  ┌{bar}┐")
    print(f"  │  {name:<44}│")
    print(f"  ├{bar}┤")
    print(f"  │  MAE   : ₹{mae:>14,.0f}{'':>18}│")
    print(f"  │  RMSE  : ₹{rmse:>14,.0f}{'':>18}│")
    print(f"  │  MAPE  : {mape:>14.2f}%{'':>17}│")
    print(f"  │  R²    : {r2:>14.4f}{'':>18}│")
    print(f"  └{bar}┘")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'preds': preds}


# ═══════════════════════════════════════════════════════════════════════════
# 5. MODELS
# ═══════════════════════════════════════════════════════════════════════════

def train_random_forest(X_train, y_train):
    print("\n🌲 Training Random Forest baseline…")
    model = RandomForestRegressor(
        n_estimators=300, max_depth=20,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def train_xgboost_default(X_train, y_train):
    print("\n⚡ Training XGBoost (default params)…")
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS):
    """
    Optuna searches the XGBoost hyperparameter space using 5-fold CV.
    Minimises MAE — the most interpretable metric for price prediction.
    """
    print(f"\n🔬 Fine-tuning XGBoost with Optuna ({n_trials} trials)…")
    print("  Each trial = 5-fold CV. This takes a few minutes…\n")

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int  ('n_estimators',      200, 1000),
            'learning_rate':     trial.suggest_float('learning_rate',      0.005, 0.3,  log=True),
            'max_depth':         trial.suggest_int  ('max_depth',          3, 12),
            'min_child_weight':  trial.suggest_int  ('min_child_weight',   1, 10),
            'subsample':         trial.suggest_float('subsample',          0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel',  0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',          1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',         1e-8, 10.0, log=True),
            'gamma':             trial.suggest_float('gamma',              0.0, 5.0),
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
        }
        scores = cross_val_score(
            XGBRegressor(**params), X_train, y_train,
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return -scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\n  Best CV MAE : ₹{study.best_value:,.0f}")
    print(f"  Best params found:")
    for k, v in study.best_params.items():
        print(f"    {k:<25} : {v}")

    # Train final model on FULL training set with best params
    print("\n  Training final model on full training set…")
    best_model = XGBRegressor(
        **study.best_params,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    best_model.fit(X_train, y_train)
    print("  ✓ Done")
    return best_model, study


# ═══════════════════════════════════════════════════════════════════════════
# 6. PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def save_plots(y_test, results, kept_features, importance_all, study=None):
    print("\n📊 Saving plots…")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('AutoVault — Car Price Prediction Results', fontsize=14, fontweight='bold')
    PAL = {
        'Random Forest':      '#2196F3',
        'XGBoost Default':    '#FF9800',
        'XGBoost Fine-tuned': '#4CAF50'
    }
    names = list(results.keys())

    # 1. Predicted vs Actual
    ax  = axes[0, 0]
    p   = results['XGBoost Fine-tuned']['preds']
    ax.scatter(y_test / 1e6, p / 1e6, alpha=0.25, s=6, color='#4CAF50')
    mv  = max(float(y_test.max()), float(p.max())) / 1e6
    ax.plot([0, mv], [0, mv], 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual (₹M)'); ax.set_ylabel('Predicted (₹M)')
    ax.set_title('Predicted vs Actual (Fine-tuned XGB)'); ax.legend(fontsize=8)

    # 2. Residuals
    ax  = axes[0, 1]
    res = (p - y_test) / 1e6
    ax.scatter(p / 1e6, res, alpha=0.25, s=6, color='#9C27B0')
    ax.axhline(0, color='red', lw=1.5, linestyle='--')
    ax.set_xlabel('Predicted (₹M)'); ax.set_ylabel('Residual (₹M)')
    ax.set_title('Residuals')

    # 3. MAE comparison
    ax   = axes[0, 2]
    maes = [results[m]['mae'] / 1e6 for m in names]
    bars = ax.bar(names, maes, color=[PAL[m] for m in names], alpha=0.85)
    for b, v in zip(bars, maes):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'₹{v:.2f}M', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('MAE (₹M)'); ax.set_title('Model MAE Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 4. R² comparison
    ax  = axes[1, 0]
    r2s = [results[m]['r2'] for m in names]
    bars= ax.bar(names, r2s, color=[PAL[m] for m in names], alpha=0.85)
    ax.set_ylim(0, 1.05)
    for b, v in zip(bars, r2s):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('R²'); ax.set_title('Model R² Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 5. Feature importance — green = kept, red = dropped
    ax     = axes[1, 1]
    imp_s  = importance_all.sort_values()
    colors = ['#4CAF50' if f in kept_features else '#f44336' for f in imp_s.index]
    ax.barh(imp_s.index, imp_s.values, color=colors, alpha=0.85)
    ax.axvline(IMPORTANCE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold ({IMPORTANCE_THRESHOLD})')
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance\n(green = kept  |  red = auto-dropped)')
    ax.legend(fontsize=7)

    # 6. Optuna history
    ax = axes[1, 2]
    if study:
        vals = [t.value / 1e6 for t in study.trials if t.value is not None]
        best = pd.Series(vals).cummin().values
        ax.plot(vals, alpha=0.35, color='#FF9800', label='Trial MAE')
        ax.plot(best, color='#4CAF50', lw=2,      label='Best so far')
        ax.set_xlabel('Trial'); ax.set_ylabel('MAE (₹M)')
        ax.set_title('Optuna Optimisation History'); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No study data', ha='center',
                va='center', transform=ax.transAxes)

    plt.tight_layout()
    out = os.path.join(MODEL_DIR, 'training_results.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════
# 7. INFERENCE  —  use this after training to predict a single car
# ═══════════════════════════════════════════════════════════════════════════

def predict_price(brand, year, kms_covered, fuel_type, transmission, tier):
    """
    Predict the listed price of a single car.

    Parameters
    ----------
    brand        : str    e.g. 'BMW', 'Ferrari', 'Maruti'
    year         : int    e.g. 2022
    kms_covered  : float  e.g. 35000.0
    fuel_type    : str    'Petrol' | 'Diesel' | 'Electric' | 'CNG' | 'Hybrid'
    transmission : str    'Automatic' | 'Manual'
    tier         : str    'luxury' | 'standard'

    Returns
    -------
    float  —  predicted price in ₹
    """
    model     = joblib.load(os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    encoders  = joblib.load(os.path.join(MODEL_DIR, 'encoders.pkl'))
    feat_cols = joblib.load(os.path.join(MODEL_DIR, 'feature_columns.pkl'))

    brand_target = encoders['brand_target']
    global_mean  = encoders.get('brand_global_mean', 1_000_000)

    def safe_le(le, val):
        val = str(val).strip()
        return int(le.transform([val])[0]) if val in le.classes_ else 0

    row = {
        'brand_price_mean': brand_target.get(str(brand).strip(), global_mean),
        'year':             int(year),
        'log_kms':          float(np.log1p(kms_covered)),
        'fuel_type_enc':    safe_le(encoders['fuel_type'],    fuel_type),
        'transmission_enc': safe_le(encoders['transmission'], transmission),
        'tier_enc':         safe_le(encoders['tier'],         tier),
    }

    X = pd.DataFrame([row])[feat_cols]
    return float(model.predict(X)[0])


# ═══════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    # ── Load ──────────────────────────────────────────────────────────────
    df = load_data()

    # ── Clean + engineer features ─────────────────────────────────────────
    df, encoders, brand_encoding = clean_and_engineer(df)
    encoders['brand_global_mean'] = float(df['price'].mean())

    # ── Build feature matrix ──────────────────────────────────────────────
    X_all = df[CANDIDATE_FEATURES]
    y_all = df['price']

    # ── Train / test split ────────────────────────────────────────────────
    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"\n📦 Train : {len(X_train_full):,} rows")
    print(f"   Test  : {len(X_test_full):,} rows")

    # ── Auto feature selection ────────────────────────────────────────────
    kept, dropped, importance_all = select_features(X_train_full, y_train)

    # Use only the kept features from here on
    X_train = X_train_full[kept]
    X_test  = X_test_full[kept]

    # ── Train all three models ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING  ({len(kept)} features: {kept})")
    print(f"{'='*60}")

    rf_model         = train_random_forest(X_train, y_train)
    xgb_default      = train_xgboost_default(X_train, y_train)
    xgb_tuned, study = fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS)

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  EVALUATION  (held-out test set)")
    print(f"{'='*60}")
    results = {
        'Random Forest':      evaluate(rf_model,    X_test, y_test, "Random Forest"),
        'XGBoost Default':    evaluate(xgb_default, X_test, y_test, "XGBoost Default"),
        'XGBoost Fine-tuned': evaluate(xgb_tuned,   X_test, y_test, "XGBoost Fine-tuned"),
    }

    # ── Save ──────────────────────────────────────────────────────────────
    print("\n💾 Saving models…")
    joblib.dump(rf_model,    os.path.join(MODEL_DIR, 'rf_baseline.pkl'))
    joblib.dump(xgb_default, os.path.join(MODEL_DIR, 'xgb_default.pkl'))
    joblib.dump(xgb_tuned,   os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    joblib.dump(encoders,    os.path.join(MODEL_DIR, 'encoders.pkl'))
    joblib.dump(kept,        os.path.join(MODEL_DIR, 'feature_columns.pkl'))
    print(f"  ✓ Saved to: {MODEL_DIR}/")

    # ── Plots ─────────────────────────────────────────────────────────────
    save_plots(y_test, results, kept, importance_all, study)

    # ── Sample predictions ────────────────────────────────────────────────
    print(f"\n🔮 Sample Predictions:")
    print(f"  {'Brand':<16} {'Year':<6} {'Kms':>8}  {'Fuel':<10} {'Trans':<12} {'Tier':<10}  Predicted")
    print(f"  {'─'*82}")
    samples = [
        ('Ferrari',      2024, 4000,   'Petrol',  'Automatic', 'luxury'),
        ('Rolls-Royce',  2025, 1500,   'Petrol',  'Automatic', 'luxury'),
        ('BMW',          2022, 35000,  'Petrol',  'Automatic', 'standard'),
        ('Mercedes-Benz',2021, 20000,  'Diesel',  'Automatic', 'luxury'),
        ('Porsche',      2023, 8000,   'Petrol',  'Automatic', 'luxury'),
        ('Hyundai',      2020, 55000,  'Diesel',  'Manual',    'standard'),
        ('Maruti',       2018, 75000,  'Petrol',  'Manual',    'standard'),
        ('Tata',         2021, 40000,  'Petrol',  'Manual',    'standard'),
    ]
    for brand, year, kms, fuel, trans, tier in samples:
        price = predict_price(brand, year, kms, fuel, trans, tier)
        print(f"  {brand:<16} {year:<6} {kms:>8,}  {fuel:<10} {trans:<12} {tier:<10}  ₹{price:>14,.0f}")

    # ── Summary ───────────────────────────────────────────────────────────
    best = results['XGBoost Fine-tuned']
    print(f"\n{'='*60}")
    print(f"  BEST MODEL  : XGBoost Fine-tuned (Optuna {OPTUNA_TRIALS} trials)")
    print(f"  R²          : {best['r2']:.4f}  →  {best['r2']*100:.1f}% variance explained")
    print(f"  MAE         : ₹{best['mae']:,.0f}")
    print(f"  MAPE        : {best['mape']:.2f}%")
    print(f"  Features    : {kept}")
    print(f"  Auto-dropped: {dropped}")
    print(f"\n  Production files:")
    print(f"  → models/xgb_finetuned.pkl")
    print(f"  → models/encoders.pkl")
    print(f"  → models/feature_columns.pkl")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  AutoVault — Car Price Prediction

📂 Loading CSVs…
  Luxury   :  1,091 rows
  Ordinary : 13,756 rows
  Combined : 14,847 rows

🔧 Cleaning & engineering features…
  Dropped 0 rows with missing/invalid values
  Removed 283 outliers → 14,564 rows remain
  Price range: ₹135,000 – ₹32,800,000
  fuel_type: 7 classes → ['CNG', 'Diesel', 'Electric', 'Hybrid', 'LPG', 'Petrol', 'Unknown']
  transmission: 3 classes → ['Automatic', 'Manual', 'Unknown']
  tier: 2 classes → ['luxury', 'standard']
  ✓ Feature engineering complete

📦 Train : 11,651 rows
   Test  : 2,913 rows

🔍 Automatic feature selection…
  Threshold: importance >= 0.01

  Feature                  Importance  Decision
  ────────────────────────────────────────────────
  tier_enc                     0.9373  ✓  KEEP
  transmission_enc             0.0242  ✓  KEEP
  brand_price_mean             0.0120  ✓  KEEP
  year                         0.0113  ✓  KEEP
  log_kms                      0.0078  ✗  DROP (below threshold)
  fuel_type_enc 

Best trial: 13. Best value: 439766: 100%|██████████| 50/50 [05:46<00:00,  6.93s/it]



  Best CV MAE : ₹439,766
  Best params found:
    n_estimators              : 751
    learning_rate             : 0.06162816973645811
    max_depth                 : 9
    min_child_weight          : 1
    subsample                 : 0.8408923779866572
    colsample_bytree          : 0.9735709574142806
    colsample_bylevel         : 0.7751948605221033
    reg_alpha                 : 0.02280654446384315
    reg_lambda                : 3.622661233069749e-08
    gamma                     : 1.1860127584218203

  Training final model on full training set…
  ✓ Done

  EVALUATION  (held-out test set)

  ┌──────────────────────────────────────────────┐
  │  Random Forest                               │
  ├──────────────────────────────────────────────┤
  │  MAE   : ₹       414,503                  │
  │  RMSE  : ₹     1,114,044                  │
  │  MAPE  :          28.61%                 │
  │  R²    :         0.8629                  │
  └──────────────────────────────────────────────┘

 

In [6]:
"""
AutoVault — Car Price Prediction Model
========================================
Reads from:
  data/merged_datasets/merged_luxe_dataset.csv
  data/merged_datasets/merged_ordinary_dataset.csv

Features used (auto-selected by importance):
  - tier          (luxury vs standard — strongest signal)
  - log_kms       (log of km driven — better than raw kms)
  - year          (model year)
  - brand         (target-encoded — mean price per brand)
  - fuel_type     (kept only if passes importance threshold)
  - transmission  (kept only if passes importance threshold)

Features REMOVED automatically:
  - kms_covered   (redundant — log_kms captures this better)
  - car_age       (redundant — same info as year, just flipped)
  - kms_per_year  (low importance, causes noise)
  - brand_enc     (label encoding brands is meaningless — replaced by target encoding)

Models:
  1. Random Forest       — baseline
  2. XGBoost (default)   — better accuracy
  3. XGBoost Fine-tuned  — Optuna hyperparameter search (BEST)

Output → models/
  xgb_finetuned.pkl, rf_baseline.pkl,
  encoders.pkl, feature_columns.pkl

SETUP:
  pip install pandas scikit-learn xgboost optuna matplotlib seaborn joblib

RUN:
  python train_price_model.py
"""

import os
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection   import train_test_split, KFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble          import RandomForestRegressor
from xgboost                   import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
LUXE_CSV  = "../../data/merged_datasets/merged_luxe_dataset.csv"
ORD_CSV   = "../../data/merged_datasets/merged_ordinary_dataset.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

CURRENT_YEAR         = 2026
OPTUNA_TRIALS        = 50       # increase to 100+ for better tuning
TEST_SIZE            = 0.20
RANDOM_STATE         = 42
IMPORTANCE_THRESHOLD = 0.01     # features below this are auto-dropped


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# BRAND / MODEL EXTRACTION FROM TITLE
# Title format is always:  "<Brand> <Model>"
# e.g. "Mercedes-Benz AMG GLE Coupe" → brand="Mercedes-Benz", model="AMG GLE Coupe"
#      "Land Rover Range Rover"       → brand="Land Rover",    model="Range Rover"
#      "Maruti Ertiga"                → brand="Maruti",        model="Ertiga"
# Multi-word brands are matched first (longest match wins).
# ═══════════════════════════════════════════════════════════════════════════

# All known multi-word or hyphenated brands — order doesn't matter, sorted by length below
KNOWN_BRANDS = [
    'Mercedes-Benz', 'Land Rover', 'Aston Martin', 'Rolls-Royce',
    'Rolls royce', 'Mini Cooper', 'Maruti Suzuki', 'Alfa Romeo',
    'BMW', 'Audi', 'Ferrari', 'Lamborghini', 'Porsche', 'McLaren',
    'Bentley', 'Jaguar', 'Lexus', 'Toyota', 'Honda', 'Hyundai',
    'Maruti', 'Mahindra', 'Tata', 'Skoda', 'Volkswagen', 'Volvo',
    'Renault', 'Nissan', 'Ford', 'Jeep', 'Kia', 'KIA', 'MG',
    'BYD', 'Datsun', 'Fiat', 'Isuzu', 'Mercedes',
]
# Sort longest first so multi-word brands match before single-word ones
KNOWN_BRANDS_SORTED = sorted(KNOWN_BRANDS, key=len, reverse=True)

BRAND_NORMALISE = {
    'Rolls royce':   'Rolls-Royce',
    'Maruti Suzuki': 'Maruti',
    'Mercedes':      'Mercedes-Benz',
    'KIA':           'Kia',
}


def extract_brand_model(title: str):
    """
    Extract (brand, model) from a car title string.

    Returns
    -------
    (brand, model) : tuple of str
        e.g. ("Mercedes-Benz", "AMG GLE Coupe")
        e.g. ("Maruti", "Ertiga")
    """
    if not isinstance(title, str) or not title.strip():
        return 'Unknown', 'Unknown'

    title = title.strip()

    # Try matching known multi-word / hyphenated brands first
    for brand in KNOWN_BRANDS_SORTED:
        if title.lower().startswith(brand.lower()):
            model = title[len(brand):].strip()
            brand = BRAND_NORMALISE.get(brand, brand)
            return brand, model if model else 'Unknown'

    # Fallback: first word is brand, rest is model
    parts = title.split(maxsplit=1)
    brand = BRAND_NORMALISE.get(parts[0], parts[0])
    model = parts[1] if len(parts) > 1 else 'Unknown'
    return brand, model


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

def load_data() -> pd.DataFrame:
    print("=" * 60)
    print("  AutoVault — Car Price Prediction")
    print("=" * 60)
    print("\n📂 Loading CSVs…")

    lux = pd.read_csv(LUXE_CSV)
    lux = lux.drop(columns=[c for c in lux.columns if 'Unnamed' in c], errors='ignore')
    lux = lux.rename(columns={
        'Title': 'title', 'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })
    lux['tier'] = 'luxury'

    ord_ = pd.read_csv(ORD_CSV)
    ord_ = ord_.drop(columns=[c for c in ord_.columns if 'Unnamed' in c], errors='ignore')
    ord_ = ord_.rename(columns={
        'Title': 'title', 'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })
    ord_['tier'] = 'standard'

    needed = ['title', 'brand', 'price', 'kms_covered', 'year',
              'fuel_type', 'transmission', 'tier']
    df = pd.concat([lux[needed], ord_[needed]], ignore_index=True)

    # ── Extract brand & model from title ──────────────────────────────────
    # This is the source of truth — fills null brands and adds a clean
    # model column that was not available before.
    extracted = df['title'].apply(extract_brand_model)
    df['brand_from_title'] = extracted.apply(lambda x: x[0])
    df['model']            = extracted.apply(lambda x: x[1])

    # Fill null/unknown brand with brand extracted from title
    df['brand'] = df['brand'].fillna('').str.strip()
    df['brand'] = df.apply(
        lambda r: r['brand_from_title']
                  if r['brand'] in ('', 'Unknown', 'nan') or pd.isna(r['brand'])
                  else r['brand'],
        axis=1
    )

    print(f"  Luxury   : {len(lux):>6,} rows")
    print(f"  Ordinary : {len(ord_):>6,} rows")
    print(f"  Combined : {len(df):>6,} rows")

    # Show extraction samples
    print(f"\n  Sample title → brand / model extraction:")
    sample = df[['title', 'brand', 'model']].drop_duplicates('title').head(8)
    for _, row in sample.iterrows():
        print(f"    {row['title']:<35} → brand={row['brand']:<20} model={row['model']}")

    return df


# ═══════════════════════════════════════════════════════════════════════════
# 2. CLEAN & ENGINEER FEATURES
# ═══════════════════════════════════════════════════════════════════════════

def clean_and_engineer(df: pd.DataFrame):
    print("\n🔧 Cleaning & engineering features…")
    df = df.copy()

    # Drop rows with missing target / key numerics
    before = len(df)
    df = df.dropna(subset=['price', 'kms_covered', 'year'])
    df = df[df['price'] > 0]
    df = df[df['kms_covered'] >= 0]
    print(f"  Dropped {before - len(df):,} rows with missing/invalid values")

    # Normalise brands (brand was already filled from title in load_data)
    brand_map = BRAND_NORMALISE  # reuse same map
    fuel_map  = {'CNG & Hybrids': 'CNG'}

    df['brand']        = df['brand'].str.strip().replace(brand_map).fillna('Unknown')
    df['model']        = df['model'].fillna('Unknown').str.strip()
    df['fuel_type']    = df['fuel_type'].fillna('Unknown').str.strip().replace(fuel_map)
    df['transmission'] = df['transmission'].fillna('Unknown').str.strip()
    df['tier']         = df['tier'].fillna('standard').str.strip()

    # Cast numerics
    df['year']        = pd.to_numeric(df['year'],        errors='coerce').fillna(2020).astype(int)
    df['kms_covered'] = pd.to_numeric(df['kms_covered'], errors='coerce').fillna(0).astype(float)
    df['price']       = pd.to_numeric(df['price'],       errors='coerce')
    df = df.dropna(subset=['price'])

    # Remove outliers (1st–99th percentile per tier)
    before = len(df)
    parts = []
    for tier in df['tier'].unique():
        sub = df[df['tier'] == tier].copy()
        lo, hi = sub['price'].quantile([0.01, 0.99])
        parts.append(sub[(sub['price'] >= lo) & (sub['price'] <= hi)])
    df = pd.concat(parts, ignore_index=True)
    print(f"  Removed {before - len(df):,} outliers → {len(df):,} rows remain")
    print(f"  Price range: ₹{df['price'].min():,.0f} – ₹{df['price'].max():,.0f}")
    print(f"  Unique brands extracted from title: {df['brand'].nunique()}")
    print(f"  Unique models extracted from title: {df['model'].nunique()}")

    # ── Target-encode brand (mean price per brand) ─────────────────────
    # Much better than label encoding — Ferrari >> Maruti in price space
    brand_mean = df.groupby('brand')['price'].mean()
    df['brand_price_mean'] = df['brand'].map(brand_mean)
    brand_encoding = brand_mean.to_dict()

    # ── Target-encode model (mean price per model) ─────────────────────
    # "911 Turbo S" should be worth more than "911 Carrera" — this captures it
    model_mean = df.groupby('model')['price'].mean()
    df['model_price_mean'] = df['model'].map(model_mean)
    model_encoding = model_mean.to_dict()

    # ── Label encode remaining categoricals ───────────────────────────
    encoders = {
        'brand_target': brand_encoding,
        'model_target': model_encoding,
    }
    for col in ['fuel_type', 'transmission', 'tier']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  {col}: {len(le.classes_)} classes → {list(le.classes_)}")

    # ── log_kms: log-transform compresses the huge kms range ──────────
    df['log_kms'] = np.log1p(df['kms_covered'])

    print(f"  ✓ Feature engineering complete")
    return df, encoders, brand_encoding


# ═══════════════════════════════════════════════════════════════════════════
# 3. AUTOMATIC FEATURE SELECTION
#    Train a quick XGBoost on ALL candidate features, measure each
#    feature's importance score, auto-drop anything below threshold.
# ═══════════════════════════════════════════════════════════════════════════

CANDIDATE_FEATURES = [
    'brand_price_mean',   # target-encoded brand  (e.g. Ferrari >> Maruti)
    'model_price_mean',   # target-encoded model  (e.g. 911 Turbo >> 718 Boxster)
    'year',               # model year of the car
    'log_kms',            # log-transformed km driven
    'fuel_type_enc',      # petrol / diesel / electric / cng / hybrid
    'transmission_enc',   # manual / automatic
    'tier_enc',           # luxury vs standard (strongest single feature)
]


def select_features(X_train: pd.DataFrame, y_train: pd.Series):
    print("\n🔍 Automatic feature selection…")
    print(f"  Threshold: importance >= {IMPORTANCE_THRESHOLD}")

    # Quick XGBoost to measure importance
    probe = XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    probe.fit(X_train, y_train)

    importance = pd.Series(
        probe.feature_importances_,
        index=CANDIDATE_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  {'Feature':<22} {'Importance':>12}  Decision")
    print(f"  {'─' * 48}")
    kept, dropped = [], []
    for feat, imp in importance.items():
        decision = '✓  KEEP' if imp >= IMPORTANCE_THRESHOLD else '✗  DROP (below threshold)'
        print(f"  {feat:<22} {imp:>12.4f}  {decision}")
        if imp >= IMPORTANCE_THRESHOLD:
            kept.append(feat)
        else:
            dropped.append(feat)

    print(f"\n  → Kept    ({len(kept)}): {kept}")
    print(f"  → Dropped ({len(dropped)}): {dropped}")
    return kept, dropped, importance


# ═══════════════════════════════════════════════════════════════════════════
# 4. EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(model, X_test, y_test, name="Model"):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = math.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / y_test.clip(1))) * 100

    bar = '─' * 46
    print(f"\n  ┌{bar}┐")
    print(f"  │  {name:<44}│")
    print(f"  ├{bar}┤")
    print(f"  │  MAE   : ₹{mae:>14,.0f}{'':>18}│")
    print(f"  │  RMSE  : ₹{rmse:>14,.0f}{'':>18}│")
    print(f"  │  MAPE  : {mape:>14.2f}%{'':>17}│")
    print(f"  │  R²    : {r2:>14.4f}{'':>18}│")
    print(f"  └{bar}┘")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'preds': preds}


# ═══════════════════════════════════════════════════════════════════════════
# 5. MODELS
# ═══════════════════════════════════════════════════════════════════════════

def train_random_forest(X_train, y_train):
    print("\n🌲 Training Random Forest baseline…")
    model = RandomForestRegressor(
        n_estimators=300, max_depth=20,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def train_xgboost_default(X_train, y_train):
    print("\n⚡ Training XGBoost (default params)…")
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS):
    """
    Optuna searches the XGBoost hyperparameter space using 5-fold CV.
    Minimises MAE — the most interpretable metric for price prediction.
    """
    print(f"\n🔬 Fine-tuning XGBoost with Optuna ({n_trials} trials)…")
    print("  Each trial = 5-fold CV. This takes a few minutes…\n")

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int  ('n_estimators',      200, 1000),
            'learning_rate':     trial.suggest_float('learning_rate',      0.005, 0.3,  log=True),
            'max_depth':         trial.suggest_int  ('max_depth',          3, 12),
            'min_child_weight':  trial.suggest_int  ('min_child_weight',   1, 10),
            'subsample':         trial.suggest_float('subsample',          0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel',  0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',          1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',         1e-8, 10.0, log=True),
            'gamma':             trial.suggest_float('gamma',              0.0, 5.0),
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
        }
        scores = cross_val_score(
            XGBRegressor(**params), X_train, y_train,
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return -scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\n  Best CV MAE : ₹{study.best_value:,.0f}")
    print(f"  Best params found:")
    for k, v in study.best_params.items():
        print(f"    {k:<25} : {v}")

    # Train final model on FULL training set with best params
    print("\n  Training final model on full training set…")
    best_model = XGBRegressor(
        **study.best_params,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    best_model.fit(X_train, y_train)
    print("  ✓ Done")
    return best_model, study


# ═══════════════════════════════════════════════════════════════════════════
# 6. PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def save_plots(y_test, results, kept_features, importance_all, study=None):
    print("\n📊 Saving plots…")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('AutoVault — Car Price Prediction Results', fontsize=14, fontweight='bold')
    PAL = {
        'Random Forest':      '#2196F3',
        'XGBoost Default':    '#FF9800',
        'XGBoost Fine-tuned': '#4CAF50'
    }
    names = list(results.keys())

    # 1. Predicted vs Actual
    ax  = axes[0, 0]
    p   = results['XGBoost Fine-tuned']['preds']
    ax.scatter(y_test / 1e6, p / 1e6, alpha=0.25, s=6, color='#4CAF50')
    mv  = max(float(y_test.max()), float(p.max())) / 1e6
    ax.plot([0, mv], [0, mv], 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual (₹M)'); ax.set_ylabel('Predicted (₹M)')
    ax.set_title('Predicted vs Actual (Fine-tuned XGB)'); ax.legend(fontsize=8)

    # 2. Residuals
    ax  = axes[0, 1]
    res = (p - y_test) / 1e6
    ax.scatter(p / 1e6, res, alpha=0.25, s=6, color='#9C27B0')
    ax.axhline(0, color='red', lw=1.5, linestyle='--')
    ax.set_xlabel('Predicted (₹M)'); ax.set_ylabel('Residual (₹M)')
    ax.set_title('Residuals')

    # 3. MAE comparison
    ax   = axes[0, 2]
    maes = [results[m]['mae'] / 1e6 for m in names]
    bars = ax.bar(names, maes, color=[PAL[m] for m in names], alpha=0.85)
    for b, v in zip(bars, maes):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'₹{v:.2f}M', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('MAE (₹M)'); ax.set_title('Model MAE Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 4. R² comparison
    ax  = axes[1, 0]
    r2s = [results[m]['r2'] for m in names]
    bars= ax.bar(names, r2s, color=[PAL[m] for m in names], alpha=0.85)
    ax.set_ylim(0, 1.05)
    for b, v in zip(bars, r2s):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('R²'); ax.set_title('Model R² Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 5. Feature importance — green = kept, red = dropped
    ax     = axes[1, 1]
    imp_s  = importance_all.sort_values()
    colors = ['#4CAF50' if f in kept_features else '#f44336' for f in imp_s.index]
    ax.barh(imp_s.index, imp_s.values, color=colors, alpha=0.85)
    ax.axvline(IMPORTANCE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold ({IMPORTANCE_THRESHOLD})')
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance\n(green = kept  |  red = auto-dropped)')
    ax.legend(fontsize=7)

    # 6. Optuna history
    ax = axes[1, 2]
    if study:
        vals = [t.value / 1e6 for t in study.trials if t.value is not None]
        best = pd.Series(vals).cummin().values
        ax.plot(vals, alpha=0.35, color='#FF9800', label='Trial MAE')
        ax.plot(best, color='#4CAF50', lw=2,      label='Best so far')
        ax.set_xlabel('Trial'); ax.set_ylabel('MAE (₹M)')
        ax.set_title('Optuna Optimisation History'); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No study data', ha='center',
                va='center', transform=ax.transAxes)

    plt.tight_layout()
    out = os.path.join(MODEL_DIR, 'training_results.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════
# 7. INFERENCE  —  use this after training to predict a single car
# ═══════════════════════════════════════════════════════════════════════════

def predict_price(brand, model, year, kms_covered, fuel_type, transmission, tier):
    """
    Predict the listed price of a single car.

    Parameters
    ----------
    brand        : str    e.g. 'BMW', 'Ferrari', 'Maruti'
    model        : str    e.g. 'X7', '488 GTB', 'Ertiga'  ← extracted from title
    year         : int    e.g. 2022
    kms_covered  : float  e.g. 35000.0
    fuel_type    : str    'Petrol' | 'Diesel' | 'Electric' | 'CNG' | 'Hybrid'
    transmission : str    'Automatic' | 'Manual'
    tier         : str    'luxury' | 'standard'

    Returns
    -------
    float  —  predicted price in ₹

    Tip: use extract_brand_model(title) to get brand & model from a raw title string.
    """
    mdl       = joblib.load(os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    encoders  = joblib.load(os.path.join(MODEL_DIR, 'encoders.pkl'))
    feat_cols = joblib.load(os.path.join(MODEL_DIR, 'feature_columns.pkl'))

    brand_target = encoders['brand_target']
    model_target = encoders.get('model_target', {})
    global_mean  = encoders.get('brand_global_mean', 1_000_000)
    model_global = encoders.get('model_global_mean', global_mean)

    def safe_le(le, val):
        val = str(val).strip()
        return int(le.transform([val])[0]) if val in le.classes_ else 0

    row = {
        'brand_price_mean': brand_target.get(str(brand).strip(), global_mean),
        'model_price_mean': model_target.get(str(model).strip(), model_global),
        'year':             int(year),
        'log_kms':          float(np.log1p(kms_covered)),
        'fuel_type_enc':    safe_le(encoders['fuel_type'],    fuel_type),
        'transmission_enc': safe_le(encoders['transmission'], transmission),
        'tier_enc':         safe_le(encoders['tier'],         tier),
    }

    X = pd.DataFrame([row])[feat_cols]
    return float(mdl.predict(X)[0])


# ═══════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    # ── Load ──────────────────────────────────────────────────────────────
    df = load_data()

    # ── Clean + engineer features ─────────────────────────────────────────
    df, encoders, brand_encoding = clean_and_engineer(df)
    encoders['brand_global_mean'] = float(df['price'].mean())
    encoders['model_global_mean'] = float(df['price'].mean())

    # ── Build feature matrix ──────────────────────────────────────────────
    X_all = df[CANDIDATE_FEATURES]
    y_all = df['price']

    # ── Train / test split ────────────────────────────────────────────────
    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"\n📦 Train : {len(X_train_full):,} rows")
    print(f"   Test  : {len(X_test_full):,} rows")

    # ── Auto feature selection ────────────────────────────────────────────
    kept, dropped, importance_all = select_features(X_train_full, y_train)

    # Use only the kept features from here on
    X_train = X_train_full[kept]
    X_test  = X_test_full[kept]

    # ── Train all three models ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING  ({len(kept)} features: {kept})")
    print(f"{'='*60}")

    rf_model         = train_random_forest(X_train, y_train)
    xgb_default      = train_xgboost_default(X_train, y_train)
    xgb_tuned, study = fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS)

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  EVALUATION  (held-out test set)")
    print(f"{'='*60}")
    results = {
        'Random Forest':      evaluate(rf_model,    X_test, y_test, "Random Forest"),
        'XGBoost Default':    evaluate(xgb_default, X_test, y_test, "XGBoost Default"),
        'XGBoost Fine-tuned': evaluate(xgb_tuned,   X_test, y_test, "XGBoost Fine-tuned"),
    }

    # ── Save ──────────────────────────────────────────────────────────────
    print("\n💾 Saving models…")
    joblib.dump(rf_model,    os.path.join(MODEL_DIR, 'rf_baseline.pkl'))
    joblib.dump(xgb_default, os.path.join(MODEL_DIR, 'xgb_default.pkl'))
    joblib.dump(xgb_tuned,   os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    joblib.dump(encoders,    os.path.join(MODEL_DIR, 'encoders.pkl'))
    joblib.dump(kept,        os.path.join(MODEL_DIR, 'feature_columns.pkl'))
    print(f"  ✓ Saved to: {MODEL_DIR}/")

    # ── Plots ─────────────────────────────────────────────────────────────
    save_plots(y_test, results, kept, importance_all, study)

    # ── Sample predictions ────────────────────────────────────────────────
    print(f"\n🔮 Sample Predictions (brand & model extracted from title):")
    print(f"  {'Title':<35} {'Year':<6} {'Kms':>8}  {'Fuel':<10} {'Tier':<10}  Predicted")
    print(f"  {'─'*88}")
    samples = [
        ('Ferrari 488 GTB',           2024, 4000,   'Petrol',  'Automatic', 'luxury'),
        ('Rolls-Royce Ghost',         2025, 1500,   'Petrol',  'Automatic', 'luxury'),
        ('BMW X7',                    2022, 35000,  'Petrol',  'Automatic', 'standard'),
        ('Mercedes-Benz AMG GLE',     2021, 20000,  'Diesel',  'Automatic', 'luxury'),
        ('Porsche 911',               2023, 8000,   'Petrol',  'Automatic', 'luxury'),
        ('Hyundai Creta',             2020, 55000,  'Diesel',  'Manual',    'standard'),
        ('Maruti Ertiga',             2018, 75000,  'Petrol',  'Manual',    'standard'),
        ('Tata Nexon',                2021, 40000,  'Petrol',  'Manual',    'standard'),
        ('Land Rover Range Rover',    2023, 12000,  'Diesel',  'Automatic', 'luxury'),
        ('Lamborghini Urus SE',       2024, 3000,   'Petrol',  'Automatic', 'luxury'),
    ]
    for title, year, kms, fuel, trans, tier in samples:
        brand, model = extract_brand_model(title)
        price = predict_price(brand, model, year, kms, fuel, trans, tier)
        print(f"  {title:<35} {year:<6} {kms:>8,}  {fuel:<10} {tier:<10}  ₹{price:>14,.0f}")

    # ── Summary ───────────────────────────────────────────────────────────
    best = results['XGBoost Fine-tuned']
    print(f"\n{'='*60}")
    print(f"  BEST MODEL  : XGBoost Fine-tuned (Optuna {OPTUNA_TRIALS} trials)")
    print(f"  R²          : {best['r2']:.4f}  →  {best['r2']*100:.1f}% variance explained")
    print(f"  MAE         : ₹{best['mae']:,.0f}")
    print(f"  MAPE        : {best['mape']:.2f}%")
    print(f"  Features    : {kept}")
    print(f"  Auto-dropped: {dropped}")
    print(f"\n  Production files:")
    print(f"  → models/xgb_finetuned_1.pkl")
    print(f"  → models/encoders_1.pkl")
    print(f"  → models/feature_columns_1.pkl")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  AutoVault — Car Price Prediction

📂 Loading CSVs…
  Luxury   :  1,091 rows
  Ordinary : 13,756 rows
  Combined : 14,847 rows

  Sample title → brand / model extraction:
    2022/23 LAND ROVER DEFENDER 2.0 HSE 110 → brand=2022/23              model=LAND ROVER DEFENDER 2.0 HSE 110
    2015/16 LAMBORGHINI HURACAN COUPE LP 610-4 → brand=2015/16              model=LAMBORGHINI HURACAN COUPE LP 610-4
    2022/23 MINI COOPER SE 3 DOOR       → brand=2022/23              model=MINI COOPER SE 3 DOOR
    2024/25 TOYOTA LAND CRUISER 300     → brand=2024/25              model=TOYOTA LAND CRUISER 300
    2022 MERCEDES BENZ EQS 53 AMG 4MATIC+ → brand=2022                 model=MERCEDES BENZ EQS 53 AMG 4MATIC+
    2025 RANGE ROVER VELAR DYN HSE      → brand=2025                 model=RANGE ROVER VELAR DYN HSE
    2024/25 AUDI Q3 40 TFSI             → brand=2024/25              model=AUDI Q3 40 TFSI
    2024/26 MERCEDES BENZ CLE 300 CABRIOLET → brand=2024/26              model=MERCEDES BENZ CLE 300 CA

Best trial: 9. Best value: 303566: 100%|██████████| 50/50 [04:44<00:00,  5.69s/it]



  Best CV MAE : ₹303,566
  Best params found:
    n_estimators              : 943
    learning_rate             : 0.27361010508539807
    max_depth                 : 11
    min_child_weight          : 1
    subsample                 : 0.6899413936838258
    colsample_bytree          : 0.46633408222444406
    colsample_bylevel         : 0.7875169673833506
    reg_alpha                 : 2.9152895018922766e-07
    reg_lambda                : 5.884838345243317e-07
    gamma                     : 4.89600405223819

  Training final model on full training set…
  ✓ Done

  EVALUATION  (held-out test set)

  ┌──────────────────────────────────────────────┐
  │  Random Forest                               │
  ├──────────────────────────────────────────────┤
  │  MAE   : ₹       259,652                  │
  │  RMSE  : ₹       669,817                  │
  │  MAPE  :          20.84%                 │
  │  R²    :         0.9504                  │
  └──────────────────────────────────────────────┘

In [7]:
"""
AutoVault — Car Price Prediction Model
========================================
Reads from:
  data/merged_datasets/merged_luxe_dataset.csv
  data/merged_datasets/merged_ordinary_dataset.csv

Features used (auto-selected by importance):
  - log_kms       (log of km driven — better than raw kms)
  - year          (model year)
  - brand         (target-encoded — mean price per brand)
  - fuel_type     (kept only if passes importance threshold)
  - transmission  (kept only if passes importance threshold)

Features REMOVED automatically:
  - kms_covered   (redundant — log_kms captures this better)
  - car_age       (redundant — same info as year, just flipped)
  - kms_per_year  (low importance, causes noise)
  - brand_enc     (label encoding brands is meaningless — replaced by target encoding)

Models:
  1. Random Forest       — baseline
  2. XGBoost (default)   — better accuracy
  3. XGBoost Fine-tuned  — Optuna hyperparameter search (BEST)

Output → models/
  xgb_finetuned.pkl, rf_baseline.pkl,
  encoders.pkl, feature_columns.pkl

SETUP:
  pip install pandas scikit-learn xgboost optuna matplotlib seaborn joblib

RUN:
  python train_price_model.py
"""

import os
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection   import train_test_split, KFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble          import RandomForestRegressor
from xgboost                   import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
ORD_CSV   = "../../data/merged_datasets/merged_ordinary_dataset.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

CURRENT_YEAR         = 2026
OPTUNA_TRIALS        = 50       # increase to 100+ for better tuning
TEST_SIZE            = 0.20
RANDOM_STATE         = 42
IMPORTANCE_THRESHOLD = 0.01     # features below this are auto-dropped


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# BRAND / MODEL EXTRACTION FROM TITLE
# Title format is always:  "<Brand> <Model>"
# e.g. "Mercedes-Benz AMG GLE Coupe" → brand="Mercedes-Benz", model="AMG GLE Coupe"
#      "Land Rover Range Rover"       → brand="Land Rover",    model="Range Rover"
#      "Maruti Ertiga"                → brand="Maruti",        model="Ertiga"
# Multi-word brands are matched first (longest match wins).
# ═══════════════════════════════════════════════════════════════════════════

# All known multi-word or hyphenated brands — order doesn't matter, sorted by length below
KNOWN_BRANDS = [
    'Mercedes-Benz', 'Land Rover', 'Aston Martin', 'Rolls-Royce',
    'Rolls royce', 'Mini Cooper', 'Maruti Suzuki', 'Alfa Romeo',
    'BMW', 'Audi', 'Ferrari', 'Lamborghini', 'Porsche', 'McLaren',
    'Bentley', 'Jaguar', 'Lexus', 'Toyota', 'Honda', 'Hyundai',
    'Maruti', 'Mahindra', 'Tata', 'Skoda', 'Volkswagen', 'Volvo',
    'Renault', 'Nissan', 'Ford', 'Jeep', 'Kia', 'KIA', 'MG',
    'BYD', 'Datsun', 'Fiat', 'Isuzu', 'Mercedes',
]
# Sort longest first so multi-word brands match before single-word ones
KNOWN_BRANDS_SORTED = sorted(KNOWN_BRANDS, key=len, reverse=True)

BRAND_NORMALISE = {
    'Rolls royce':   'Rolls-Royce',
    'Maruti Suzuki': 'Maruti',
    'Mercedes':      'Mercedes-Benz',
    'KIA':           'Kia',
}


def extract_brand_model(title: str):
    """
    Extract (brand, model) from a car title string.

    Returns
    -------
    (brand, model) : tuple of str
        e.g. ("Mercedes-Benz", "AMG GLE Coupe")
        e.g. ("Maruti", "Ertiga")
    """
    if not isinstance(title, str) or not title.strip():
        return 'Unknown', 'Unknown'

    title = title.strip()

    # Try matching known multi-word / hyphenated brands first
    for brand in KNOWN_BRANDS_SORTED:
        if title.lower().startswith(brand.lower()):
            model = title[len(brand):].strip()
            brand = BRAND_NORMALISE.get(brand, brand)
            return brand, model if model else 'Unknown'

    # Fallback: first word is brand, rest is model
    parts = title.split(maxsplit=1)
    brand = BRAND_NORMALISE.get(parts[0], parts[0])
    model = parts[1] if len(parts) > 1 else 'Unknown'
    return brand, model


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

def load_data() -> pd.DataFrame:
    print("=" * 60)
    print("  AutoVault — Car Price Prediction")
    print("=" * 60)
    print("\n📂 Loading CSVs…")

    # Only ordinary cars — luxury data varies too diversely for a stable model
    ord_ = pd.read_csv(ORD_CSV)
    ord_ = ord_.drop(columns=[c for c in ord_.columns if 'Unnamed' in c], errors='ignore')
    ord_ = ord_.rename(columns={
        'Title': 'title', 'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })

    needed = ['title', 'brand', 'price', 'kms_covered', 'year', 'fuel_type', 'transmission']
    df = ord_[needed].copy()

    # ── Extract brand & model from title ──────────────────────────────────
    extracted = df['title'].apply(extract_brand_model)
    df['brand_from_title'] = extracted.apply(lambda x: x[0])
    df['model']            = extracted.apply(lambda x: x[1])

    # Fill null/unknown brand with brand extracted from title
    df['brand'] = df['brand'].fillna('').str.strip()
    df['brand'] = df.apply(
        lambda r: r['brand_from_title']
                  if r['brand'] in ('', 'Unknown', 'nan') or pd.isna(r['brand'])
                  else r['brand'],
        axis=1
    )

    print(f"  Ordinary : {len(df):>6,} rows")

    # Show extraction samples
    print(f"\n  Sample title → brand / model extraction:")
    sample = df[['title', 'brand', 'model']].drop_duplicates('title').head(8)
    for _, row in sample.iterrows():
        print(f"    {row['title']:<35} → brand={row['brand']:<20} model={row['model']}")

    return df


# ═══════════════════════════════════════════════════════════════════════════
# 2. CLEAN & ENGINEER FEATURES
# ═══════════════════════════════════════════════════════════════════════════

def clean_and_engineer(df: pd.DataFrame):
    print("\n🔧 Cleaning & engineering features…")
    df = df.copy()

    # Drop rows with missing target / key numerics
    before = len(df)
    df = df.dropna(subset=['price', 'kms_covered', 'year'])
    df = df[df['price'] > 0]
    df = df[df['kms_covered'] >= 0]
    print(f"  Dropped {before - len(df):,} rows with missing/invalid values")

    # Normalise brands (brand was already filled from title in load_data)
    brand_map = BRAND_NORMALISE  # reuse same map
    fuel_map  = {'CNG & Hybrids': 'CNG'}

    df['brand']        = df['brand'].str.strip().replace(brand_map).fillna('Unknown')
    df['model']        = df['model'].fillna('Unknown').str.strip()
    df['fuel_type']    = df['fuel_type'].fillna('Unknown').str.strip().replace(fuel_map)
    df['transmission'] = df['transmission'].fillna('Unknown').str.strip()

    # Cast numerics
    df['year']        = pd.to_numeric(df['year'],        errors='coerce').fillna(2020).astype(int)
    df['kms_covered'] = pd.to_numeric(df['kms_covered'], errors='coerce').fillna(0).astype(float)
    df['price']       = pd.to_numeric(df['price'],       errors='coerce')
    df = df.dropna(subset=['price'])

    # Remove outliers (1st–99th percentile on full dataset)
    before = len(df)
    lo, hi = df['price'].quantile([0.01, 0.99])
    df = df[(df['price'] >= lo) & (df['price'] <= hi)].copy()
    print(f"  Removed {before - len(df):,} outliers → {len(df):,} rows remain")
    print(f"  Price range: ₹{df['price'].min():,.0f} – ₹{df['price'].max():,.0f}")
    print(f"  Unique brands extracted from title: {df['brand'].nunique()}")
    print(f"  Unique models extracted from title: {df['model'].nunique()}")

    # ── Target-encode brand (mean price per brand) ─────────────────────
    # Much better than label encoding — Ferrari >> Maruti in price space
    brand_mean = df.groupby('brand')['price'].mean()
    df['brand_price_mean'] = df['brand'].map(brand_mean)
    brand_encoding = brand_mean.to_dict()

    # ── Target-encode model (mean price per model) ─────────────────────
    # "911 Turbo S" should be worth more than "911 Carrera" — this captures it
    model_mean = df.groupby('model')['price'].mean()
    df['model_price_mean'] = df['model'].map(model_mean)
    model_encoding = model_mean.to_dict()

    # ── Label encode remaining categoricals ───────────────────────────
    encoders = {
        'brand_target': brand_encoding,
        'model_target': model_encoding,
    }
    for col in ['fuel_type', 'transmission']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  {col}: {len(le.classes_)} classes → {list(le.classes_)}")

    # ── log_kms: log-transform compresses the huge kms range ──────────
    df['log_kms'] = np.log1p(df['kms_covered'])

    print(f"  ✓ Feature engineering complete")
    return df, encoders, brand_encoding


# ═══════════════════════════════════════════════════════════════════════════
# 3. AUTOMATIC FEATURE SELECTION
#    Train a quick XGBoost on ALL candidate features, measure each
#    feature's importance score, auto-drop anything below threshold.
# ═══════════════════════════════════════════════════════════════════════════

CANDIDATE_FEATURES = [
    'brand_price_mean',   # target-encoded brand  (e.g. Ferrari >> Maruti)
    'model_price_mean',   # target-encoded model  (e.g. 911 Turbo >> 718 Boxster)
    'year',               # model year of the car
    'log_kms',            # log-transformed km driven
    'fuel_type_enc',      # petrol / diesel / electric / cng / hybrid
    'transmission_enc',   # manual / automatic
]


def select_features(X_train: pd.DataFrame, y_train: pd.Series):
    print("\n🔍 Automatic feature selection…")
    print(f"  Threshold: importance >= {IMPORTANCE_THRESHOLD}")

    # Quick XGBoost to measure importance
    probe = XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    probe.fit(X_train, y_train)

    importance = pd.Series(
        probe.feature_importances_,
        index=CANDIDATE_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  {'Feature':<22} {'Importance':>12}  Decision")
    print(f"  {'─' * 48}")
    kept, dropped = [], []
    for feat, imp in importance.items():
        decision = '✓  KEEP' if imp >= IMPORTANCE_THRESHOLD else '✗  DROP (below threshold)'
        print(f"  {feat:<22} {imp:>12.4f}  {decision}")
        if imp >= IMPORTANCE_THRESHOLD:
            kept.append(feat)
        else:
            dropped.append(feat)

    print(f"\n  → Kept    ({len(kept)}): {kept}")
    print(f"  → Dropped ({len(dropped)}): {dropped}")
    return kept, dropped, importance


# ═══════════════════════════════════════════════════════════════════════════
# 4. EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(model, X_test, y_test, name="Model"):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = math.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / y_test.clip(1))) * 100

    bar = '─' * 46
    print(f"\n  ┌{bar}┐")
    print(f"  │  {name:<44}│")
    print(f"  ├{bar}┤")
    print(f"  │  MAE   : ₹{mae:>14,.0f}{'':>18}│")
    print(f"  │  RMSE  : ₹{rmse:>14,.0f}{'':>18}│")
    print(f"  │  MAPE  : {mape:>14.2f}%{'':>17}│")
    print(f"  │  R²    : {r2:>14.4f}{'':>18}│")
    print(f"  └{bar}┘")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'preds': preds}


# ═══════════════════════════════════════════════════════════════════════════
# 5. MODELS
# ═══════════════════════════════════════════════════════════════════════════

def train_random_forest(X_train, y_train):
    print("\n🌲 Training Random Forest baseline…")
    model = RandomForestRegressor(
        n_estimators=300, max_depth=20,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def train_xgboost_default(X_train, y_train):
    print("\n⚡ Training XGBoost (default params)…")
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS):
    """
    Optuna searches the XGBoost hyperparameter space using 5-fold CV.
    Minimises MAE — the most interpretable metric for price prediction.
    """
    print(f"\n🔬 Fine-tuning XGBoost with Optuna ({n_trials} trials)…")
    print("  Each trial = 5-fold CV. This takes a few minutes…\n")

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int  ('n_estimators',      200, 1000),
            'learning_rate':     trial.suggest_float('learning_rate',      0.005, 0.3,  log=True),
            'max_depth':         trial.suggest_int  ('max_depth',          3, 12),
            'min_child_weight':  trial.suggest_int  ('min_child_weight',   1, 10),
            'subsample':         trial.suggest_float('subsample',          0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel',  0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',          1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',         1e-8, 10.0, log=True),
            'gamma':             trial.suggest_float('gamma',              0.0, 5.0),
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
        }
        scores = cross_val_score(
            XGBRegressor(**params), X_train, y_train,
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return -scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\n  Best CV MAE : ₹{study.best_value:,.0f}")
    print(f"  Best params found:")
    for k, v in study.best_params.items():
        print(f"    {k:<25} : {v}")

    # Train final model on FULL training set with best params
    print("\n  Training final model on full training set…")
    best_model = XGBRegressor(
        **study.best_params,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    best_model.fit(X_train, y_train)
    print("  ✓ Done")
    return best_model, study


# ═══════════════════════════════════════════════════════════════════════════
# 6. PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def save_plots(y_test, results, kept_features, importance_all, study=None):
    print("\n📊 Saving plots…")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('AutoVault — Car Price Prediction Results', fontsize=14, fontweight='bold')
    PAL = {
        'Random Forest':      '#2196F3',
        'XGBoost Default':    '#FF9800',
        'XGBoost Fine-tuned': '#4CAF50'
    }
    names = list(results.keys())

    # 1. Predicted vs Actual
    ax  = axes[0, 0]
    p   = results['XGBoost Fine-tuned']['preds']
    ax.scatter(y_test / 1e6, p / 1e6, alpha=0.25, s=6, color='#4CAF50')
    mv  = max(float(y_test.max()), float(p.max())) / 1e6
    ax.plot([0, mv], [0, mv], 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual (₹M)'); ax.set_ylabel('Predicted (₹M)')
    ax.set_title('Predicted vs Actual (Fine-tuned XGB)'); ax.legend(fontsize=8)

    # 2. Residuals
    ax  = axes[0, 1]
    res = (p - y_test) / 1e6
    ax.scatter(p / 1e6, res, alpha=0.25, s=6, color='#9C27B0')
    ax.axhline(0, color='red', lw=1.5, linestyle='--')
    ax.set_xlabel('Predicted (₹M)'); ax.set_ylabel('Residual (₹M)')
    ax.set_title('Residuals')

    # 3. MAE comparison
    ax   = axes[0, 2]
    maes = [results[m]['mae'] / 1e6 for m in names]
    bars = ax.bar(names, maes, color=[PAL[m] for m in names], alpha=0.85)
    for b, v in zip(bars, maes):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'₹{v:.2f}M', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('MAE (₹M)'); ax.set_title('Model MAE Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 4. R² comparison
    ax  = axes[1, 0]
    r2s = [results[m]['r2'] for m in names]
    bars= ax.bar(names, r2s, color=[PAL[m] for m in names], alpha=0.85)
    ax.set_ylim(0, 1.05)
    for b, v in zip(bars, r2s):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('R²'); ax.set_title('Model R² Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 5. Feature importance — green = kept, red = dropped
    ax     = axes[1, 1]
    imp_s  = importance_all.sort_values()
    colors = ['#4CAF50' if f in kept_features else '#f44336' for f in imp_s.index]
    ax.barh(imp_s.index, imp_s.values, color=colors, alpha=0.85)
    ax.axvline(IMPORTANCE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold ({IMPORTANCE_THRESHOLD})')
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance\n(green = kept  |  red = auto-dropped)')
    ax.legend(fontsize=7)

    # 6. Optuna history
    ax = axes[1, 2]
    if study:
        vals = [t.value / 1e6 for t in study.trials if t.value is not None]
        best = pd.Series(vals).cummin().values
        ax.plot(vals, alpha=0.35, color='#FF9800', label='Trial MAE')
        ax.plot(best, color='#4CAF50', lw=2,      label='Best so far')
        ax.set_xlabel('Trial'); ax.set_ylabel('MAE (₹M)')
        ax.set_title('Optuna Optimisation History'); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No study data', ha='center',
                va='center', transform=ax.transAxes)

    plt.tight_layout()
    out = os.path.join(MODEL_DIR, 'training_results.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════
# 7. INFERENCE  —  use this after training to predict a single car
# ═══════════════════════════════════════════════════════════════════════════

def predict_price(brand, model, year, kms_covered, fuel_type, transmission):
    """
    Predict the listed price of a single car.

    Parameters
    ----------
    brand        : str    e.g. 'BMW', 'Ferrari', 'Maruti'
    model        : str    e.g. 'X7', '488 GTB', 'Ertiga'  ← extracted from title
    year         : int    e.g. 2022
    kms_covered  : float  e.g. 35000.0
    fuel_type    : str    'Petrol' | 'Diesel' | 'Electric' | 'CNG' | 'Hybrid'
    transmission : str    'Automatic' | 'Manual'

    Returns
    -------
    float  —  predicted price in ₹

    Tip: use extract_brand_model(title) to get brand & model from a raw title string.
    """
    mdl       = joblib.load(os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    encoders  = joblib.load(os.path.join(MODEL_DIR, 'encoders.pkl'))
    feat_cols = joblib.load(os.path.join(MODEL_DIR, 'feature_columns.pkl'))

    brand_target = encoders['brand_target']
    model_target = encoders.get('model_target', {})
    global_mean  = encoders.get('brand_global_mean', 1_000_000)
    model_global = encoders.get('model_global_mean', global_mean)

    def safe_le(le, val):
        val = str(val).strip()
        return int(le.transform([val])[0]) if val in le.classes_ else 0

    row = {
        'brand_price_mean': brand_target.get(str(brand).strip(), global_mean),
        'model_price_mean': model_target.get(str(model).strip(), model_global),
        'year':             int(year),
        'log_kms':          float(np.log1p(kms_covered)),
        'fuel_type_enc':    safe_le(encoders['fuel_type'],    fuel_type),
        'transmission_enc': safe_le(encoders['transmission'], transmission),
    }

    X = pd.DataFrame([row])[feat_cols]
    return float(mdl.predict(X)[0])


# ═══════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    # ── Load ──────────────────────────────────────────────────────────────
    df = load_data()

    # ── Clean + engineer features ─────────────────────────────────────────
    df, encoders, brand_encoding = clean_and_engineer(df)
    encoders['brand_global_mean'] = float(df['price'].mean())
    encoders['model_global_mean'] = float(df['price'].mean())

    # ── Build feature matrix ──────────────────────────────────────────────
    X_all = df[CANDIDATE_FEATURES]
    y_all = df['price']

    # ── Train / test split ────────────────────────────────────────────────
    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"\n📦 Train : {len(X_train_full):,} rows")
    print(f"   Test  : {len(X_test_full):,} rows")

    # ── Auto feature selection ────────────────────────────────────────────
    kept, dropped, importance_all = select_features(X_train_full, y_train)

    # Use only the kept features from here on
    X_train = X_train_full[kept]
    X_test  = X_test_full[kept]

    # ── Train all three models ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING  ({len(kept)} features: {kept})")
    print(f"{'='*60}")

    rf_model         = train_random_forest(X_train, y_train)
    xgb_default      = train_xgboost_default(X_train, y_train)
    xgb_tuned, study = fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS)

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  EVALUATION  (held-out test set)")
    print(f"{'='*60}")
    results = {
        'Random Forest':      evaluate(rf_model,    X_test, y_test, "Random Forest"),
        'XGBoost Default':    evaluate(xgb_default, X_test, y_test, "XGBoost Default"),
        'XGBoost Fine-tuned': evaluate(xgb_tuned,   X_test, y_test, "XGBoost Fine-tuned"),
    }

    # ── Save ──────────────────────────────────────────────────────────────
    print("\n💾 Saving models…")
    joblib.dump(rf_model,    os.path.join(MODEL_DIR, 'rf_baseline.pkl'))
    joblib.dump(xgb_default, os.path.join(MODEL_DIR, 'xgb_default.pkl'))
    joblib.dump(xgb_tuned,   os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    joblib.dump(encoders,    os.path.join(MODEL_DIR, 'encoders.pkl'))
    joblib.dump(kept,        os.path.join(MODEL_DIR, 'feature_columns.pkl'))
    print(f"  ✓ Saved to: {MODEL_DIR}/")

    # ── Plots ─────────────────────────────────────────────────────────────
    save_plots(y_test, results, kept, importance_all, study)

    # ── Sample predictions ────────────────────────────────────────────────
    print(f"\n🔮 Sample Predictions (brand & model extracted from title):")
    print(f"  {'Title':<35} {'Year':<6} {'Kms':>8}  {'Fuel':<10} {'Trans':<12}  Predicted")
    print(f"  {'─'*88}")
    samples = [
        ('Ferrari 488 GTB',           2024, 4000,   'Petrol',  'Automatic'),
        ('Rolls-Royce Ghost',         2025, 1500,   'Petrol',  'Automatic'),
        ('BMW X7',                    2022, 35000,  'Petrol',  'Automatic'),
        ('Mercedes-Benz AMG GLE',     2021, 20000,  'Diesel',  'Automatic'),
        ('Porsche 911',               2023, 8000,   'Petrol',  'Automatic'),
        ('Hyundai Creta',             2020, 55000,  'Diesel',  'Manual'),
        ('Maruti Ertiga',             2018, 75000,  'Petrol',  'Manual'),
        ('Tata Nexon',                2021, 40000,  'Petrol',  'Manual'),
        ('Land Rover Range Rover',    2023, 12000,  'Diesel',  'Automatic'),
        ('Lamborghini Urus SE',       2024, 3000,   'Petrol',  'Automatic'),
    ]
    for title, year, kms, fuel, trans in samples:
        brand, model = extract_brand_model(title)
        price = predict_price(brand, model, year, kms, fuel, trans)
        print(f"  {title:<35} {year:<6} {kms:>8,}  {fuel:<10} {trans:<12}  ₹{price:>14,.0f}")

    # ── Summary ───────────────────────────────────────────────────────────
    best = results['XGBoost Fine-tuned']
    print(f"\n{'='*60}")
    print(f"  BEST MODEL  : XGBoost Fine-tuned (Optuna {OPTUNA_TRIALS} trials)")
    print(f"  R²          : {best['r2']:.4f}  →  {best['r2']*100:.1f}% variance explained")
    print(f"  MAE         : ₹{best['mae']:,.0f}")
    print(f"  MAPE        : {best['mape']:.2f}%")
    print(f"  Features    : {kept}")
    print(f"  Auto-dropped: {dropped}")
    print(f"\n  Production files:")
    print(f"  → models/xgb_finetuned.pkl")
    print(f"  → models/encoders.pkl")
    print(f"  → models/feature_columns.pkl")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  AutoVault — Car Price Prediction

📂 Loading CSVs…
  Ordinary : 13,756 rows

  Sample title → brand / model extraction:
    Maruti Ertiga                       → brand=Maruti               model=Ertiga
    Maruti Vitara Brezza                → brand=Maruti               model=Vitara Brezza
    Maruti BREZZA                       → brand=Maruti               model=BREZZA
    Maruti IGNIS                        → brand=Maruti               model=IGNIS
    Maruti Swift                        → brand=Maruti               model=Swift
    Maruti Ciaz                         → brand=Maruti               model=Ciaz
    Maruti XL6                          → brand=Maruti               model=XL6
    Mahindra XUV300                     → brand=Mahindra             model=XUV300

🔧 Cleaning & engineering features…
  Dropped 0 rows with missing/invalid values
  Removed 269 outliers → 13,487 rows remain
  Price range: ₹135,000 – ₹6,975,000
  Unique brands extracted from title: 27
  Unique models extr

Best trial: 30. Best value: 101162: 100%|██████████| 50/50 [07:58<00:00,  9.57s/it]



  Best CV MAE : ₹101,162
  Best params found:
    n_estimators              : 951
    learning_rate             : 0.03973309031057621
    max_depth                 : 11
    min_child_weight          : 4
    subsample                 : 0.6910073727668641
    colsample_bytree          : 0.7228201130542857
    colsample_bylevel         : 0.9927463722133207
    reg_alpha                 : 1.386681110544258e-08
    reg_lambda                : 0.0014253093593179739
    gamma                     : 3.601158196102089

  Training final model on full training set…
  ✓ Done

  EVALUATION  (held-out test set)

  ┌──────────────────────────────────────────────┐
  │  Random Forest                               │
  ├──────────────────────────────────────────────┤
  │  MAE   : ₹       104,406                  │
  │  RMSE  : ₹       203,098                  │
  │  MAPE  :          10.42%                 │
  │  R²    :         0.9618                  │
  └──────────────────────────────────────────────┘


In [9]:
"""
AutoVault — Car Price Prediction Model
========================================
Reads from:
  data/merged_datasets/merged_luxe_dataset.csv
  data/merged_datasets/merged_ordinary_dataset.csv

Features used (auto-selected by importance):
  - log_kms       (log of km driven — better than raw kms)
  - year          (model year)
  - brand         (target-encoded — mean price per brand)
  - fuel_type     (kept only if passes importance threshold)
  - transmission  (kept only if passes importance threshold)

Features REMOVED automatically:
  - kms_covered   (redundant — log_kms captures this better)
  - car_age       (redundant — same info as year, just flipped)
  - kms_per_year  (low importance, causes noise)
  - brand_enc     (label encoding brands is meaningless — replaced by target encoding)

Models:
  1. Random Forest       — baseline
  2. XGBoost (default)   — better accuracy
  3. XGBoost Fine-tuned  — Optuna hyperparameter search (BEST)

Output → models/
  xgb_finetuned.pkl, rf_baseline.pkl,
  encoders.pkl, feature_columns.pkl

SETUP:
  pip install pandas scikit-learn xgboost optuna matplotlib seaborn joblib

RUN:
  python train_price_model.py
"""

import os
import math
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection   import train_test_split, KFold, cross_val_score
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble          import RandomForestRegressor
from xgboost                   import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════
ORD_CSV   = "../../data/merged_datasets/merged_ordinary_dataset.csv"
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

CURRENT_YEAR         = 2026
OPTUNA_TRIALS        = 50       # increase to 100+ for better tuning
TEST_SIZE            = 0.20
RANDOM_STATE         = 42
IMPORTANCE_THRESHOLD = 0.01     # features below this are auto-dropped


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════════════════════
# BRAND / MODEL EXTRACTION FROM TITLE
# Title format is always:  "<Brand> <Model>"
# e.g. "Mercedes-Benz AMG GLE Coupe" → brand="Mercedes-Benz", model="AMG GLE Coupe"
#      "Land Rover Range Rover"       → brand="Land Rover",    model="Range Rover"
#      "Maruti Ertiga"                → brand="Maruti",        model="Ertiga"
# Multi-word brands are matched first (longest match wins).
# ═══════════════════════════════════════════════════════════════════════════

# All known multi-word or hyphenated brands — order doesn't matter, sorted by length below
KNOWN_BRANDS = [
    'Mercedes-Benz', 'Land Rover', 'Aston Martin', 'Rolls-Royce',
    'Rolls royce', 'Mini Cooper', 'Maruti Suzuki', 'Alfa Romeo',
    'BMW', 'Audi', 'Ferrari', 'Lamborghini', 'Porsche', 'McLaren',
    'Bentley', 'Jaguar', 'Lexus', 'Toyota', 'Honda', 'Hyundai',
    'Maruti', 'Mahindra', 'Tata', 'Skoda', 'Volkswagen', 'Volvo',
    'Renault', 'Nissan', 'Ford', 'Jeep', 'Kia', 'KIA', 'MG',
    'BYD', 'Datsun', 'Fiat', 'Isuzu', 'Mercedes',
]
# Sort longest first so multi-word brands match before single-word ones
KNOWN_BRANDS_SORTED = sorted(KNOWN_BRANDS, key=len, reverse=True)

BRAND_NORMALISE = {
    'Rolls royce':   'Rolls-Royce',
    'Maruti Suzuki': 'Maruti',
    'Mercedes':      'Mercedes-Benz',
    'KIA':           'Kia',
}


def extract_brand_model(title: str):
    """
    Extract (brand, model) from a car title string.

    Returns
    -------
    (brand, model) : tuple of str
        e.g. ("Mercedes-Benz", "AMG GLE Coupe")
        e.g. ("Maruti", "Ertiga")
    """
    if not isinstance(title, str) or not title.strip():
        return 'Unknown', 'Unknown'

    title = title.strip()

    # Try matching known multi-word / hyphenated brands first
    for brand in KNOWN_BRANDS_SORTED:
        if title.lower().startswith(brand.lower()):
            model = title[len(brand):].strip()
            brand = BRAND_NORMALISE.get(brand, brand)
            return brand, model if model else 'Unknown'

    # Fallback: first word is brand, rest is model
    parts = title.split(maxsplit=1)
    brand = BRAND_NORMALISE.get(parts[0], parts[0])
    model = parts[1] if len(parts) > 1 else 'Unknown'
    return brand, model


# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD & MERGE CSVs
# ═══════════════════════════════════════════════════════════════════════════

def load_data() -> pd.DataFrame:
    print("=" * 60)
    print("  AutoVault — Car Price Prediction")
    print("=" * 60)
    print("\n📂 Loading CSVs…")

    # Only ordinary cars — luxury data varies too diversely for a stable model
    ord_ = pd.read_csv(ORD_CSV)
    ord_ = ord_.drop(columns=[c for c in ord_.columns if 'Unnamed' in c], errors='ignore')
    ord_ = ord_.rename(columns={
        'Title': 'title', 'Brand': 'brand', 'Price': 'price',
        'Kms Covered': 'kms_covered', 'Year': 'year',
        'Fuel Type': 'fuel_type', 'Type': 'transmission',
    })

    needed = ['title', 'brand', 'price', 'kms_covered', 'year', 'fuel_type', 'transmission']
    df = ord_[needed].copy()

    # ── Extract brand & model from title ──────────────────────────────────
    extracted = df['title'].apply(extract_brand_model)
    df['brand_from_title'] = extracted.apply(lambda x: x[0])
    df['model']            = extracted.apply(lambda x: x[1])

    # Fill null/unknown brand with brand extracted from title
    df['brand'] = df['brand'].fillna('').str.strip()
    df['brand'] = df.apply(
        lambda r: r['brand_from_title']
                  if r['brand'] in ('', 'Unknown', 'nan') or pd.isna(r['brand'])
                  else r['brand'],
        axis=1
    )

    print(f"  Ordinary : {len(df):>6,} rows")

    # Show extraction samples
    print(f"\n  Sample title → brand / model extraction:")
    sample = df[['title', 'brand', 'model']].drop_duplicates('title').head(8)
    for _, row in sample.iterrows():
        print(f"    {row['title']:<35} → brand={row['brand']:<20} model={row['model']}")

    return df


# ═══════════════════════════════════════════════════════════════════════════
# 2. CLEAN & ENGINEER FEATURES
# ═══════════════════════════════════════════════════════════════════════════

def clean_and_engineer(df: pd.DataFrame):
    print("\n🔧 Cleaning & engineering features…")
    df = df.copy()

    # Drop rows with missing target / key numerics
    before = len(df)
    df = df.dropna(subset=['price', 'kms_covered', 'year'])
    df = df[df['price'] > 0]
    df = df[df['kms_covered'] >= 0]
    print(f"  Dropped {before - len(df):,} rows with missing/invalid values")

    # Normalise brands (brand was already filled from title in load_data)
    brand_map = BRAND_NORMALISE  # reuse same map
    fuel_map  = {'CNG & Hybrids': 'CNG'}

    df['brand']        = df['brand'].str.strip().replace(brand_map).fillna('Unknown')
    df['model']        = df['model'].fillna('Unknown').str.strip()
    df['fuel_type']    = df['fuel_type'].fillna('Unknown').str.strip().replace(fuel_map)
    df['transmission'] = df['transmission'].fillna('Unknown').str.strip()

    # Cast numerics
    df['year']        = pd.to_numeric(df['year'],        errors='coerce').fillna(2020).astype(int)
    df['kms_covered'] = pd.to_numeric(df['kms_covered'], errors='coerce').fillna(0).astype(float)
    df['price']       = pd.to_numeric(df['price'],       errors='coerce')
    df = df.dropna(subset=['price'])

    # Remove outliers (1st–99th percentile on full dataset)
    before = len(df)
    lo, hi = df['price'].quantile([0.01, 0.99])
    df = df[(df['price'] >= lo) & (df['price'] <= hi)].copy()
    print(f"  Removed {before - len(df):,} outliers → {len(df):,} rows remain")
    print(f"  Price range: ₹{df['price'].min():,.0f} – ₹{df['price'].max():,.0f}")
    print(f"  Unique brands extracted from title: {df['brand'].nunique()}")
    print(f"  Unique models extracted from title: {df['model'].nunique()}")

    # ── Target-encode brand (mean price per brand) ─────────────────────
    # Much better than label encoding — Ferrari >> Maruti in price space
    brand_mean = df.groupby('brand')['price'].mean()
    df['brand_price_mean'] = df['brand'].map(brand_mean)
    brand_encoding = brand_mean.to_dict()

    # ── Label encode remaining categoricals ───────────────────────────
    encoders = {
        'brand_target': brand_encoding,
    }
    for col in ['fuel_type', 'transmission']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
        print(f"  {col}: {len(le.classes_)} classes → {list(le.classes_)}")

    # ── log_kms: log-transform compresses the huge kms range ──────────
    df['log_kms'] = np.log1p(df['kms_covered'])

    print(f"  ✓ Feature engineering complete")
    return df, encoders, brand_encoding


# ═══════════════════════════════════════════════════════════════════════════
# 3. AUTOMATIC FEATURE SELECTION
#    Train a quick XGBoost on ALL candidate features, measure each
#    feature's importance score, auto-drop anything below threshold.
# ═══════════════════════════════════════════════════════════════════════════

CANDIDATE_FEATURES = [
    'brand_price_mean',   # target-encoded brand  (e.g. Ferrari >> Maruti)
    'year',               # model year of the car
    'log_kms',            # log-transformed km driven
    'fuel_type_enc',      # petrol / diesel / electric / cng / hybrid
    'transmission_enc',   # manual / automatic
]


def select_features(X_train: pd.DataFrame, y_train: pd.Series):
    print("\n🔍 Automatic feature selection…")
    print(f"  Threshold: importance >= {IMPORTANCE_THRESHOLD}")

    # Quick XGBoost to measure importance
    probe = XGBRegressor(
        n_estimators=300, max_depth=6,
        learning_rate=0.1, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    )
    probe.fit(X_train, y_train)

    importance = pd.Series(
        probe.feature_importances_,
        index=CANDIDATE_FEATURES
    ).sort_values(ascending=False)

    print(f"\n  {'Feature':<22} {'Importance':>12}  Decision")
    print(f"  {'─' * 48}")
    kept, dropped = [], []
    for feat, imp in importance.items():
        decision = '✓  KEEP' if imp >= IMPORTANCE_THRESHOLD else '✗  DROP (below threshold)'
        print(f"  {feat:<22} {imp:>12.4f}  {decision}")
        if imp >= IMPORTANCE_THRESHOLD:
            kept.append(feat)
        else:
            dropped.append(feat)

    print(f"\n  → Kept    ({len(kept)}): {kept}")
    print(f"  → Dropped ({len(dropped)}): {dropped}")
    return kept, dropped, importance


# ═══════════════════════════════════════════════════════════════════════════
# 4. EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

def evaluate(model, X_test, y_test, name="Model"):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = math.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    mape  = np.mean(np.abs((y_test - preds) / y_test.clip(1))) * 100

    bar = '─' * 46
    print(f"\n  ┌{bar}┐")
    print(f"  │  {name:<44}│")
    print(f"  ├{bar}┤")
    print(f"  │  MAE   : ₹{mae:>14,.0f}{'':>18}│")
    print(f"  │  RMSE  : ₹{rmse:>14,.0f}{'':>18}│")
    print(f"  │  MAPE  : {mape:>14.2f}%{'':>17}│")
    print(f"  │  R²    : {r2:>14.4f}{'':>18}│")
    print(f"  └{bar}┘")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'preds': preds}


# ═══════════════════════════════════════════════════════════════════════════
# 5. MODELS
# ═══════════════════════════════════════════════════════════════════════════

def train_random_forest(X_train, y_train):
    print("\n🌲 Training Random Forest baseline…")
    model = RandomForestRegressor(
        n_estimators=300, max_depth=20,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def train_xgboost_default(X_train, y_train):
    print("\n⚡ Training XGBoost (default params)…")
    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    model.fit(X_train, y_train)
    print("  ✓ Done")
    return model


def fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS):
    """
    Optuna searches the XGBoost hyperparameter space using 5-fold CV.
    Minimises MAE — the most interpretable metric for price prediction.
    """
    print(f"\n🔬 Fine-tuning XGBoost with Optuna ({n_trials} trials)…")
    print("  Each trial = 5-fold CV. This takes a few minutes…\n")

    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int  ('n_estimators',      200, 1000),
            'learning_rate':     trial.suggest_float('learning_rate',      0.005, 0.3,  log=True),
            'max_depth':         trial.suggest_int  ('max_depth',          3, 12),
            'min_child_weight':  trial.suggest_int  ('min_child_weight',   1, 10),
            'subsample':         trial.suggest_float('subsample',          0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',   0.4, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel',  0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',          1e-8, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',         1e-8, 10.0, log=True),
            'gamma':             trial.suggest_float('gamma',              0.0, 5.0),
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': 0,
        }
        scores = cross_val_score(
            XGBRegressor(**params), X_train, y_train,
            cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring='neg_mean_absolute_error', n_jobs=-1
        )
        return -scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print(f"\n  Best CV MAE : ₹{study.best_value:,.0f}")
    print(f"  Best params found:")
    for k, v in study.best_params.items():
        print(f"    {k:<25} : {v}")

    # Train final model on FULL training set with best params
    print("\n  Training final model on full training set…")
    best_model = XGBRegressor(
        **study.best_params,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    best_model.fit(X_train, y_train)
    print("  ✓ Done")
    return best_model, study


# ═══════════════════════════════════════════════════════════════════════════
# 6. PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def save_plots(y_test, results, kept_features, importance_all, study=None):
    print("\n📊 Saving plots…")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('AutoVault — Car Price Prediction Results', fontsize=14, fontweight='bold')
    PAL = {
        'Random Forest':      '#2196F3',
        'XGBoost Default':    '#FF9800',
        'XGBoost Fine-tuned': '#4CAF50'
    }
    names = list(results.keys())

    # 1. Predicted vs Actual
    ax  = axes[0, 0]
    p   = results['XGBoost Fine-tuned']['preds']
    ax.scatter(y_test / 1e6, p / 1e6, alpha=0.25, s=6, color='#4CAF50')
    mv  = max(float(y_test.max()), float(p.max())) / 1e6
    ax.plot([0, mv], [0, mv], 'r--', lw=1.5, label='Perfect')
    ax.set_xlabel('Actual (₹M)'); ax.set_ylabel('Predicted (₹M)')
    ax.set_title('Predicted vs Actual (Fine-tuned XGB)'); ax.legend(fontsize=8)

    # 2. Residuals
    ax  = axes[0, 1]
    res = (p - y_test) / 1e6
    ax.scatter(p / 1e6, res, alpha=0.25, s=6, color='#9C27B0')
    ax.axhline(0, color='red', lw=1.5, linestyle='--')
    ax.set_xlabel('Predicted (₹M)'); ax.set_ylabel('Residual (₹M)')
    ax.set_title('Residuals')

    # 3. MAE comparison
    ax   = axes[0, 2]
    maes = [results[m]['mae'] / 1e6 for m in names]
    bars = ax.bar(names, maes, color=[PAL[m] for m in names], alpha=0.85)
    for b, v in zip(bars, maes):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
                f'₹{v:.2f}M', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('MAE (₹M)'); ax.set_title('Model MAE Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 4. R² comparison
    ax  = axes[1, 0]
    r2s = [results[m]['r2'] for m in names]
    bars= ax.bar(names, r2s, color=[PAL[m] for m in names], alpha=0.85)
    ax.set_ylim(0, 1.05)
    for b, v in zip(bars, r2s):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('R²'); ax.set_title('Model R² Comparison')
    ax.set_xticklabels(names, rotation=10, fontsize=8)

    # 5. Feature importance — green = kept, red = dropped
    ax     = axes[1, 1]
    imp_s  = importance_all.sort_values()
    colors = ['#4CAF50' if f in kept_features else '#f44336' for f in imp_s.index]
    ax.barh(imp_s.index, imp_s.values, color=colors, alpha=0.85)
    ax.axvline(IMPORTANCE_THRESHOLD, color='black', lw=1.2, linestyle='--',
               label=f'Threshold ({IMPORTANCE_THRESHOLD})')
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance\n(green = kept  |  red = auto-dropped)')
    ax.legend(fontsize=7)

    # 6. Optuna history
    ax = axes[1, 2]
    if study:
        vals = [t.value / 1e6 for t in study.trials if t.value is not None]
        best = pd.Series(vals).cummin().values
        ax.plot(vals, alpha=0.35, color='#FF9800', label='Trial MAE')
        ax.plot(best, color='#4CAF50', lw=2,      label='Best so far')
        ax.set_xlabel('Trial'); ax.set_ylabel('MAE (₹M)')
        ax.set_title('Optuna Optimisation History'); ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No study data', ha='center',
                va='center', transform=ax.transAxes)

    plt.tight_layout()
    out = os.path.join(MODEL_DIR, 'training_results.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════
# 7. INFERENCE  —  use this after training to predict a single car
# ═══════════════════════════════════════════════════════════════════════════

def predict_price(brand, year, kms_covered, fuel_type, transmission):
    """
    Predict the listed price of a single car.

    Parameters
    ----------
    brand        : str    e.g. 'BMW', 'Ferrari', 'Maruti'
    year         : int    e.g. 2022
    kms_covered  : float  e.g. 35000.0
    fuel_type    : str    'Petrol' | 'Diesel' | 'Electric' | 'CNG' | 'Hybrid'
    transmission : str    'Automatic' | 'Manual'

    Returns
    -------
    float  —  predicted price in ₹

    Tip: use extract_brand_model(title) to get brand & model from a raw title string.
    """
    mdl       = joblib.load(os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    encoders  = joblib.load(os.path.join(MODEL_DIR, 'encoders.pkl'))
    feat_cols = joblib.load(os.path.join(MODEL_DIR, 'feature_columns.pkl'))

    brand_target = encoders['brand_target']
    global_mean  = encoders.get('brand_global_mean', 1_000_000)

    def safe_le(le, val):
        val = str(val).strip()
        return int(le.transform([val])[0]) if val in le.classes_ else 0

    row = {
        'brand_price_mean': brand_target.get(str(brand).strip(), global_mean),
        'year':             int(year),
        'log_kms':          float(np.log1p(kms_covered)),
        'fuel_type_enc':    safe_le(encoders['fuel_type'],    fuel_type),
        'transmission_enc': safe_le(encoders['transmission'], transmission),
    }

    X = pd.DataFrame([row])[feat_cols]
    return float(mdl.predict(X)[0])


# ═══════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    # ── Load ──────────────────────────────────────────────────────────────
    df = load_data()

    # ── Clean + engineer features ─────────────────────────────────────────
    df, encoders, brand_encoding = clean_and_engineer(df)
    encoders['brand_global_mean'] = float(df['price'].mean())

    # ── Build feature matrix ──────────────────────────────────────────────
    X_all = df[CANDIDATE_FEATURES]
    y_all = df['price']

    # ── Train / test split ────────────────────────────────────────────────
    X_train_full, X_test_full, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    print(f"\n📦 Train : {len(X_train_full):,} rows")
    print(f"   Test  : {len(X_test_full):,} rows")

    # ── Auto feature selection ────────────────────────────────────────────
    kept, dropped, importance_all = select_features(X_train_full, y_train)

    # Use only the kept features from here on
    X_train = X_train_full[kept]
    X_test  = X_test_full[kept]

    # ── Train all three models ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING  ({len(kept)} features: {kept})")
    print(f"{'='*60}")

    rf_model         = train_random_forest(X_train, y_train)
    xgb_default      = train_xgboost_default(X_train, y_train)
    xgb_tuned, study = fine_tune_xgboost(X_train, y_train, n_trials=OPTUNA_TRIALS)

    # ── Evaluate ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  EVALUATION  (held-out test set)")
    print(f"{'='*60}")
    results = {
        'Random Forest':      evaluate(rf_model,    X_test, y_test, "Random Forest"),
        'XGBoost Default':    evaluate(xgb_default, X_test, y_test, "XGBoost Default"),
        'XGBoost Fine-tuned': evaluate(xgb_tuned,   X_test, y_test, "XGBoost Fine-tuned"),
    }

    # ── Save ──────────────────────────────────────────────────────────────
    print("\n💾 Saving models…")
    joblib.dump(rf_model,    os.path.join(MODEL_DIR, 'rf_baseline.pkl'))
    joblib.dump(xgb_default, os.path.join(MODEL_DIR, 'xgb_default.pkl'))
    joblib.dump(xgb_tuned,   os.path.join(MODEL_DIR, 'xgb_finetuned.pkl'))
    joblib.dump(encoders,    os.path.join(MODEL_DIR, 'encoders.pkl'))
    joblib.dump(kept,        os.path.join(MODEL_DIR, 'feature_columns.pkl'))
    print(f"  ✓ Saved to: {MODEL_DIR}/")

    # ── Plots ─────────────────────────────────────────────────────────────
    save_plots(y_test, results, kept, importance_all, study)

    # ── Sample predictions ────────────────────────────────────────────────
    print(f"\n🔮 Sample Predictions (brand & model extracted from title):")
    print(f"  {'Title':<35} {'Year':<6} {'Kms':>8}  {'Fuel':<10} {'Trans':<12}  Predicted")
    print(f"  {'─'*88}")
    samples = [
        ('Ferrari 488 GTB',           2024, 4000,   'Petrol',  'Automatic'),
        ('Rolls-Royce Ghost',         2025, 1500,   'Petrol',  'Automatic'),
        ('BMW X7',                    2022, 35000,  'Petrol',  'Automatic'),
        ('Mercedes-Benz AMG GLE',     2021, 20000,  'Diesel',  'Automatic'),
        ('Porsche 911',               2023, 8000,   'Petrol',  'Automatic'),
        ('Hyundai Creta',             2020, 55000,  'Diesel',  'Manual'),
        ('Maruti Ertiga',             2018, 75000,  'Petrol',  'Manual'),
        ('Tata Nexon',                2021, 40000,  'Petrol',  'Manual'),
        ('Land Rover Range Rover',    2023, 12000,  'Diesel',  'Automatic'),
        ('Lamborghini Urus SE',       2024, 3000,   'Petrol',  'Automatic'),
    ]
    for title, year, kms, fuel, trans in samples:
        brand, model = extract_brand_model(title)
        price = predict_price(brand, year, kms, fuel, trans)
        print(f"  {title:<35} {year:<6} {kms:>8,}  {fuel:<10} {trans:<12}  ₹{price:>14,.0f}")

    # ── Summary ───────────────────────────────────────────────────────────
    best = results['XGBoost Fine-tuned']
    print(f"\n{'='*60}")
    print(f"  BEST MODEL  : XGBoost Fine-tuned (Optuna {OPTUNA_TRIALS} trials)")
    print(f"  R²          : {best['r2']:.4f}  →  {best['r2']*100:.1f}% variance explained")
    print(f"  MAE         : ₹{best['mae']:,.0f}")
    print(f"  MAPE        : {best['mape']:.2f}%")
    print(f"  Features    : {kept}")
    print(f"  Auto-dropped: {dropped}")
    print(f"\n  Production files:")
    print(f"  → models/xgb_finetuned.pkl")
    print(f"  → models/encoders.pkl")
    print(f"  → models/feature_columns.pkl")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

  AutoVault — Car Price Prediction

📂 Loading CSVs…
  Ordinary : 13,756 rows

  Sample title → brand / model extraction:
    Maruti Ertiga                       → brand=Maruti               model=Ertiga
    Maruti Vitara Brezza                → brand=Maruti               model=Vitara Brezza
    Maruti BREZZA                       → brand=Maruti               model=BREZZA
    Maruti IGNIS                        → brand=Maruti               model=IGNIS
    Maruti Swift                        → brand=Maruti               model=Swift
    Maruti Ciaz                         → brand=Maruti               model=Ciaz
    Maruti XL6                          → brand=Maruti               model=XL6
    Mahindra XUV300                     → brand=Mahindra             model=XUV300

🔧 Cleaning & engineering features…
  Dropped 0 rows with missing/invalid values
  Removed 269 outliers → 13,487 rows remain
  Price range: ₹135,000 – ₹6,975,000
  Unique brands extracted from title: 27
  Unique models extr

Best trial: 41. Best value: 220155: 100%|██████████| 50/50 [04:38<00:00,  5.58s/it]



  Best CV MAE : ₹220,155
  Best params found:
    n_estimators              : 994
    learning_rate             : 0.04847092219404494
    max_depth                 : 10
    min_child_weight          : 1
    subsample                 : 0.6017930706961817
    colsample_bytree          : 0.9152500509108682
    colsample_bylevel         : 0.5364825209632279
    reg_alpha                 : 0.00034221330696011765
    reg_lambda                : 0.5126904272999385
    gamma                     : 4.300785197379077

  Training final model on full training set…
  ✓ Done

  EVALUATION  (held-out test set)

  ┌──────────────────────────────────────────────┐
  │  Random Forest                               │
  ├──────────────────────────────────────────────┤
  │  MAE   : ₹       210,414                  │
  │  RMSE  : ₹       386,035                  │
  │  MAPE  :          20.82%                 │
  │  R²    :         0.8621                  │
  └──────────────────────────────────────────────┘

 